In [1]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install jsonlines datasets huggingface_hub

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

DRIVE_FOLDER = "/content/drive/MyDrive/MrWorldwide"

import json
import jsonlines
import random

TRAINING_FILE = f"{DRIVE_FOLDER}/training_data_v3.jsonl"
COMBINED_FILE = f"{DRIVE_FOLDER}/training_data_v3_combined.jsonl"

with jsonlines.open(TRAINING_FILE) as reader:
    synthetic_data = list(reader)

print(f"Loaded {len(synthetic_data)} synthetic examples")

# Verify tags
sample = synthetic_data[0]["messages"][2]["content"]
assert "<reasoning>" in sample and "</reasoning>" in sample, "ERROR: <reasoning> tags missing!"
assert "<think>" not in sample, "ERROR: old <think> tags still present!"
print("Tag verification passed!")

# SlimOrca ~5%
from datasets import load_dataset

slim_orca = load_dataset("Open-Orca/SlimOrca-Dedup", split="train")
n_orca = int(len(synthetic_data) * 0.05)
print(f"Sampling {n_orca} SlimOrca examples (~5%)")

orca_indices = random.sample(range(len(slim_orca)), n_orca)
GENERAL_SYSTEM = "You are a helpful assistant. Think step by step."
orca_examples = []

for idx in orca_indices:
    row = slim_orca[idx]
    messages = []
    for turn in row["conversations"]:
        role = turn["from"]
        if role == "system":
            messages.append({"role": "system", "content": turn["value"]})
        elif role == "human":
            messages.append({"role": "user", "content": turn["value"]})
        elif role == "gpt":
            messages.append({"role": "assistant", "content": turn["value"]})
    if len(messages) >= 2:
        if messages[0]["role"] != "system":
            messages.insert(0, {"role": "system", "content": GENERAL_SYSTEM})
        orca_examples.append({"messages": messages})

all_training_data = synthetic_data + orca_examples
random.shuffle(all_training_data)

with jsonlines.open(COMBINED_FILE, mode='w') as writer:
    writer.write_all(all_training_data)

print(f"\nTotal: {len(all_training_data)} examples")
print(f"  Synthetic: {len(synthetic_data)} ({len(synthetic_data)/len(all_training_data)*100:.1f}%)")
print(f"  SlimOrca:  {len(orca_examples)} ({len(orca_examples)/len(all_training_data)*100:.1f}%)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2472 synthetic examples
Tag verification passed!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.94k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

data/train-00000-of-00002-6d275f30fa8e14(…):   0%|          | 0.00/163M [00:00<?, ?B/s]

data/train-00001-of-00002-20da825e60baa0(…):   0%|          | 0.00/145M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363491 [00:00<?, ? examples/s]

Sampling 123 SlimOrca examples (~5%)

Total: 2595 examples
  Synthetic: 2472 (95.3%)
  SlimOrca:  123 (4.7%)


In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

print("Fresh base model loaded!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/100k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

Unsloth: Will load unsloth/deepseek-r1-distill-qwen-7b-unsloth-bnb-4bit as a legacy tokenizer.


unsloth/deepseek-r1-distill-qwen-7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Fresh base model loaded!


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable: 40,370,176 / 5,383,329,280 (0.75%)


In [6]:
from datasets import Dataset

# Verify chat template preserves <reasoning> tags
test_check = tokenizer.apply_chat_template(
    [{"role": "assistant", "content": "<reasoning>test</reasoning>\n\nvisible"}],
    tokenize=False,
    add_generation_prompt=False
)
assert "reasoning" in test_check, "CRITICAL: chat template strips <reasoning> tags!"
print(f"Chat template check passed!\nPreview: {test_check}\n")

def format_for_training(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

with jsonlines.open(COMBINED_FILE) as reader:
    all_data = list(reader)

dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_for_training, batched=True, batch_size=100)

print(f"Dataset ready: {len(dataset)} examples")
print("\n=== FORMATTED EXAMPLE PREVIEW ===")
print(dataset[0]["text"][:600])

Chat template check passed!
Preview: <｜begin▁of▁sentence｜><｜Assistant｜><reasoning>test</reasoning>

visible<｜end▁of▁sentence｜>



Map:   0%|          | 0/2595 [00:00<?, ? examples/s]

Dataset ready: 2595 examples

=== FORMATTED EXAMPLE PREVIEW ===
<｜begin▁of▁sentence｜>You are MrWorldwide, a Socratic French language tutor. You help students learn French by guiding them to discover correct answers themselves — never by giving direct corrections or translations.
 
Rules you must follow:
1. NEVER give the correct answer directly. Instead, use hints, leading questions, or mini-exercises.
2. Always reason about the student's error inside <reasoning>...</reasoning> tags. This reasoning is hidden from the student.
3. In your <reasoning> block, identify the specific error, reference the grammar rule from the provided context, and plan your teach


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=f"{DRIVE_FOLDER}/checkpoints_v3",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=50,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.3,
        seed=42,
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"VRAM: {used_memory} GB / {max_memory} GB")

print("\nStarting training...")
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2595 [00:00<?, ? examples/s]

GPU: NVIDIA A100-SXM4-40GB
VRAM: 8.137 GB / 39.494 GB

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,595 | Num Epochs = 3 | Total steps = 975
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,2.739979
50,1.173556
75,0.640233
100,0.490207
125,0.453173
150,0.451471
175,0.444479
200,0.382493
225,0.412279
250,0.383173


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MrWorldwide/checkpoints_v3/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MrWorldwide/checkpoints_v3/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MrWorldwide/checkpoints_v3/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MrWorldwide/checkpoints_v3/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MrWorldwide/checkpoints_v3/checkpoint-975/tokenizer_config.json.



Training complete!
  Time: 1227s
  Final loss: 0.4298


In [8]:
FastLanguageModel.for_inference(model)

TUTOR_SYSTEM_PROMPT = """You are MrWorldwide, a Socratic French language tutor. You help students learn French by guiding them to discover correct answers themselves — never by giving direct corrections or translations.

Rules you must follow:
1. NEVER give the correct answer directly. Instead, use hints, leading questions, or mini-exercises.
2. Always reason about the student's error inside <reasoning>...</reasoning> tags. This reasoning is hidden from the student.
3. In your <reasoning> block, identify the specific error, reference the grammar rule from the provided context, and plan your teaching strategy.
4. Your visible response should be encouraging, concise, and end with a question or exercise that guides the student toward the correct form.
5. Match your language complexity to the student's CEFR level:
   - A1: respond ~80% in English, use French only for key terms being taught
   - A2: respond ~50% English, ~50% French
   - B1: respond ~80% in French, English only for grammar terminology
   - B2: respond entirely in French
6. If the student's input has no errors, praise them and provide a follow-up exercise.
7. ONLY use French and English in your responses. Never use any other script."""

test_cases = [
    {
        "label": "A1 — être/avoir age error",
        "messages": [
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": """[CONTEXT]
Le verbe avoir sert pour la possession et pour dire l'âge. Pour exprimer l'âge on utilise la structure avoir + nombre + ans (on ne dit pas « être X ans »).
Conjugaison: j'ai, tu as, il/elle a, nous avons, vous avez, ils/elles ont
[/CONTEXT]

Je suis 20 ans et je suis un étudiant."""}
        ]
    },
    {
        "label": "A2 — passé composé wrong auxiliary",
        "messages": [
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": """[CONTEXT]
Le passé composé se forme avec l'auxiliaire être pour certains verbes de mouvement (aller, venir, arriver, partir, entrer, sortir, monter, descendre, naître, mourir, tomber, rester). Avec être, le participe passé s'accorde en genre et en nombre avec le sujet.
[/CONTEXT]

Hier, j'ai allé au cinéma avec mes amis."""}
        ]
    },
    {
        "label": "B1 — subjonctif missing",
        "messages": [
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": """[CONTEXT]
Le subjonctif présent s'emploie après il faut que, pour que, avant que, bien que. Formation: radical de la 3e personne du pluriel du présent, terminaisons: -e, -es, -e, -ions, -iez, -ent.
Venir: que je vienne, que tu viennes, qu'il vienne, que nous venions, que vous veniez, qu'ils viennent
[/CONTEXT]

Il faut que tu viens demain matin."""}
        ]
    },
    {
        "label": "A1 — no error (should praise + exercise)",
        "messages": [
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": """[CONTEXT]
Présent des verbes réguliers en -er: on enlève -er et on ajoute -e, -es, -e, -ons, -ez, -ent.
Exemple: parler → je parle, tu parles, il parle, nous parlons, vous parlez, ils parlent
[/CONTEXT]

Je parle français tous les jours."""}
        ]
    }
]

for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"TEST: {tc['label']}")
    print(f"{'='*60}")

    prompt_text = tokenizer.apply_chat_template(
        tc["messages"],
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_text += "<reasoning>\n"

    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    response = full_output[len(prompt_text):]

    has_reasoning_close = "</reasoning>" in response
    has_chinese = any('\u4e00' <= c <= '\u9fff' for c in response)

    if "</reasoning>" in response:
        parts = response.split("</reasoning>")
        reasoning = parts[0].strip()
        visible = parts[1].strip().replace("<｜end▁of▁sentence｜>", "").strip()
    else:
        reasoning = response
        visible = "[NO VISIBLE RESPONSE — </reasoning> not found]"

    print(f"\n[DIAGNOSTICS]")
    print(f"  </reasoning> present: {has_reasoning_close}")
    print(f"  Chinese chars: {has_chinese}")

    print(f"\n[REASONING TRACE (hidden)]")
    print(reasoning[:400])

    print(f"\n[VISIBLE RESPONSE (shown to student)]")
    print(visible[:400])

    print(f"\n{'—'*60}")

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST: A1 — être/avoir age error


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


[DIAGNOSTICS]
  </reasoning> present: True
  Chinese chars: False

[REASONING TRACE (hidden)]
><think>
<reasoning>
The student wrote 'Je suis 20 ans' instead of using 'j'ai'. This is a common mistake for A1 learners, as they might confuse the verbs avoir and être. According to the grammar rule, we should use 'avoir' to express age in French. I will guide the student to recognize the correct verb without directly correcting them.

[VISIBLE RESPONSE (shown to student)]
Good effort! You mentioned 'je suis un étudiant', which is correct. But remember, for age, we use a different different verb. Can you try saying your age with 'avoir' instead of 'être'? What would that look like?

————————————————————————————————————————————————————————————

TEST: A2 — passé composé wrong auxiliary


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[DIAGNOSTICS]
  </reasoning> present: True
  Chinese chars: False

[REASONING TRACE (hidden)]
><think>
<reasoning>
The student wrote 'j'ai allé' instead of 'j'ai été', which is a wrong auxiliary verb used. According to the grammar rule, with être, the correct form should be used for movement verbs. I will guide the student to recognize the error and encourage them to think about the right auxiliary without directly correcting them.

[VISIBLE RESPONSE (shown to student)]
Bien essayé ! You used 'j'ai' correctly, but remember that for movement verbs with être, we need a different form. What do you think the correct auxiliary is? Can you try to rewrite that part?

————————————————————————————————————————————————————————————

TEST: B1 — subjonctif missing


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[DIAGNOSTICS]
  </reasoning> present: True
  Chinese chars: False

[REASONING TRACE (hidden)]
><think>
<reasoning>
The student wrote 'Il faut que tu viens demain matin,' which incorrectly uses 'viens' instead of the correct conjugation for 'venir' in this context. The grammar rule specifies that the subjonctif présent is used after 'il faut que.' I will encourage the student to focus on the correct form of 'venir' in this sentence.

[VISIBLE RESPONSE (shown to student)]
Bien essayé ! Tu as utilisé 'il faut que', ce qui est correct. Mais fais attention à la conjugaison du verbe 'venir' ici. Que devrais-tu dire pour que tu viennes ? Essaie de le reformuler !

————————————————————————————————————————————————————————————

TEST: A1 — no error (should praise + exercise)

[DIAGNOSTICS]
  </reasoning> present: True
  Chinese chars: False

[REASONING TRACE (hidden)]
><think>
<reasoning>
The student said 'Je parle français tous les jours,' which is correct in meaning but misses the context of u

In [9]:
ADAPTER_PATH = f"{DRIVE_FOLDER}/mrworldwide_lora_v3"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MrWorldwide/mrworldwide_lora_v3/tokenizer_config.json.


LoRA adapter saved to /content/drive/MyDrive/MrWorldwide/mrworldwide_lora_v3


In [19]:
from huggingface_hub import login
login()

HF_USERNAME = "MihaiRusu"
HF_REPO_NAME = "mrworldwide-deepseek-r1-7b-socratic-fr"

# Re-export GGUF from saved adapter and push directly
model.push_to_hub_gguf(
    f"{HF_USERNAME}/{HF_REPO_NAME}",
    tokenizer,
    quantization_method="q4_k_m",
    token=True,
)

# Upload README
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj=f"{DRIVE_FOLDER}/README.md",
    path_in_repo="README.md",
    repo_id=f"{HF_USERNAME}/{HF_REPO_NAME}",
)

print(f"Done: https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_cv5h6k2w/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_cv5h6k2w_gguf/deepseek-r1-distill-qwen-7b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/deepseek-r1-distill-qwen-7b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_cv5h6k2w_gguf/deepseek-r1-distill-qwen-7b.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading deepseek-r1-distill-qwen-7b.Q4_K_M.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...still-qwen-7b.Q4_K_M.gguf:   2%|2         | 97.6MB / 4.68GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/MihaiRusu/mrworldwide-deepseek-r1-7b-socratic-fr
Unsloth: Cleaning up temporary files...
Done: https://huggingface.co/MihaiRusu/mrworldwide-deepseek-r1-7b-socratic-fr
